# Dataset ML — formato flat (X, Y)

Struttura:
- **X** `(N, 25)` = stato del fascio in ingresso (9) + parametri di controllo (16)
- **Y** `(N, 99)` = stati del fascio agli stage 1..11 concatenati (11 × 9)

In [ ]:
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt

# Il notebook è nella stessa cartella dei dataset
try:
    HERE = Path(__vsc_ipynb_file__).parent.resolve()
except NameError:
    HERE = Path().resolve()

train = torch.load(HERE / "dataset_base.pt", map_location="cpu", weights_only=False)
val   = torch.load(HERE / "dataset_val.pt",   map_location="cpu", weights_only=False)

X_tr, Y_tr = train["X"].numpy(), train["Y"].numpy()
X_va, Y_va = val["X"].numpy(),   val["Y"].numpy()

x_cols = train["x_cols"]   # 25 nomi colonne X
y_cols = train["y_cols"]   # 99 nomi colonne Y
markers = train["markers"] # marker TraceWin

print(f"Train  X: {X_tr.shape}   Y: {Y_tr.shape}")
print(f"Val    X: {X_va.shape}   Y: {Y_va.shape}")
print(f"\nColonne X (25): {x_cols}")
print(f"\nColonne Y (99, prime 9): {y_cols[:9]}")

## 0. Struttura dei tensori

In [ ]:
print(f"Campioni train: {len(X_tr):,}   val: {len(X_va):,}")
print()

print(f"X  {X_tr.shape}")
print(f"  {'i':>2}  {'Nome colonna':<16}  Shape")
print("  " + "─" * 38)
print("  [beam_state ingresso — stage 0]")
for i in range(9):
    print(f"  {i:>2}  {x_cols[i]:<16}  ({len(X_tr):,} × 1)")
print("  [parametri di controllo]")
for i in range(9, 25):
    print(f"  {i:>2}  {x_cols[i]:<16}  ({len(X_tr):,} × 1)")

print()
print(f"Y  {Y_tr.shape}  =  11 stage × 9 variabili")
print(f"  {'stage':>5}  {'marker':>7}  {'Shape':^14}  Colonne")
print("  " + "─" * 55)
for j in range(11):
    cols_j = y_cols[j*9 : j*9+9]
    print(f"  {j+1:>5}  {markers[j+1]:>7}  ({len(Y_tr):,} × 9)  {cols_j[0]} … {cols_j[-1]}")

In [ ]:
n_show = 300

fig, axes = plt.subplots(2, 1, figsize=(16, 9))

Xn = (X_tr[:n_show] - X_tr.min(0)) / (X_tr.max(0) - X_tr.min(0) + 1e-9)
im0 = axes[0].imshow(Xn.T, aspect='auto', cmap='viridis', interpolation='nearest')
axes[0].set_yticks(range(25))
axes[0].set_yticklabels(x_cols, fontsize=7)
axes[0].axhline(8.5, color='white', lw=2.5)
axes[0].set_xlabel(f'Campione (0…{n_show-1} di {len(X_tr):,})', fontsize=9)
axes[0].set_title(f'X  {X_tr.shape}  =  beam_state_0 (9)  +  parametri (16)', fontsize=10)
plt.colorbar(im0, ax=axes[0], label='norm [0–1]', shrink=0.7)

Yn = (Y_tr[:n_show] - Y_tr.min(0)) / (Y_tr.max(0) - Y_tr.min(0) + 1e-9)
im1 = axes[1].imshow(Yn.T, aspect='auto', cmap='plasma', interpolation='nearest')
axes[1].set_yticks([j * 9 + 4 for j in range(11)])
axes[1].set_yticklabels([f"stage {j+1}  (marker {markers[j+1]})" for j in range(11)], fontsize=7)
for j in range(1, 11):
    axes[1].axhline(j * 9 - 0.5, color='white', lw=1.5)
for vi in range(9):
    axes[1].text(n_show + 0.5, vi, x_cols[vi], fontsize=5.5, va='center', color='gray')
axes[1].set_xlabel(f'Campione (0…{n_show-1} di {len(Y_tr):,})', fontsize=9)
axes[1].set_title(f'Y  {Y_tr.shape}  =  11 stage × 9 variabili', fontsize=10)
plt.colorbar(im1, ax=axes[1], label='norm [0–1]', shrink=0.7)

plt.tight_layout()
plt.show()

## 1. Distribuzione delle colonne di X

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(16, 12))
axes = axes.flatten()
for i, (name, ax) in enumerate(zip(x_cols, axes)):
    ax.hist(X_tr[:, i], bins=60, color='steelblue' if i < 9 else 'coral',
            edgecolor='white', linewidth=0.3)
    ax.set_title(name, fontsize=7.5)
    mu, sig = X_tr[:, i].mean(), X_tr[:, i].std()
    ax.text(0.97, 0.95, f'μ={mu:.3g}\nσ={sig:.3g}',
            transform=ax.transAxes, ha='right', va='top', fontsize=6.5)
    ax.tick_params(labelsize=6)
plt.suptitle('Distribuzione colonne X  (blu = beam_state_0, arancio = parametri)', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Distribuzione delle variabili di Y per stage

In [ ]:
VAR = "npart_ratio"   # variabile da seguire lungo gli stage (cambia per vedere altre)
var_idx = x_cols[:9].index(VAR)  # indice dentro i 9 di ogni stage

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

# Stage 0 (da X)
axes[0].hist(X_tr[:, var_idx], bins=60, color='steelblue', edgecolor='white', lw=0.3)
axes[0].set_title(f'stage 0  (marker {markers[0]}) — ingresso', fontsize=8)
axes[0].set_xlabel(VAR, fontsize=7)

# Stage 1..11 (da Y)
for j in range(11):
    col_idx = j * 9 + var_idx
    axes[j + 1].hist(Y_tr[:, col_idx], bins=60, color='coral', edgecolor='white', lw=0.3)
    axes[j + 1].set_title(f'stage {j+1}  (marker {markers[j+1]})', fontsize=8)
    axes[j + 1].set_xlabel(VAR, fontsize=7)

plt.suptitle(f'Evoluzione di  "{VAR}"  da stage 0 a stage 11', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Un singolo campione: X → Y

In [ ]:
IDX = 0   # campione da visualizzare

x = X_tr[IDX]   # (25,)
y = Y_tr[IDX]   # (99,)

print(f"Campione #{IDX}")
print(f"\nX (25):")
for i, (name, val) in enumerate(zip(x_cols, x)):
    sep = "  ← beam_state_0" if i == 0 else ("  ← parametri" if i == 9 else "")
    print(f"  [{i:>2}] {name:<15}  {val:+.6g}{sep}")

print(f"\nY (99) — prime 18 colonne (stage 1 e 2):")
for i in range(18):
    print(f"  [{i:>2}] {y_cols[i]:<20}  {y[i]:+.6g}")

## 4. Train vs Val — confronto dimensioni e distribuzioni

In [ ]:
print(f"Train  {X_tr.shape}  /  {Y_tr.shape}   ({len(X_tr):,} campioni)")
print(f"Val    {X_va.shape}  /  {Y_va.shape}   ({len(X_va):,} campioni)")
print(f"Split  {len(X_tr)/(len(X_tr)+len(X_va)):.1%} train  /  {len(X_va)/(len(X_tr)+len(X_va)):.1%} val")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, label in zip(axes, [0, 9, 90], [x_cols[0], x_cols[9], y_cols[90]]):
    if col < 25:
        data_tr, data_va = X_tr[:, col], X_va[:, col]
    else:
        c = col - 25
        data_tr, data_va = Y_tr[:, c], Y_va[:, c]
    kw = dict(bins=60, edgecolor='white', linewidth=0.3, alpha=0.7)
    ax.hist(data_tr, **kw, color='steelblue', label='train')
    ax.hist(data_va, **kw, color='coral',     label='val')
    ax.set_title(label, fontsize=9)
    ax.legend(fontsize=8)
plt.suptitle('Train vs Val — distribuzioni di tre colonne rappresentative', fontsize=11)
plt.tight_layout()
plt.show()

## 5. Stage 1 — variabili di interesse

In [ ]:
# Stage 1 = prime 9 colonne di Y
S1_VARS = ['npart_ratio_s1', 'x0_s1', 'y0_s1', 'SizeX_s1', 'SizeY_s1',
           'ex_s1', 'ey_s1', "x'0_s1", "y'0_s1"]

# Verifica che siano effettivamente le prime 9 colonne di Y
print("Colonne Y stage 1:", y_cols[:9])
assert list(y_cols[:9]) == S1_VARS, "Mismatch nomi colonne!"

Y_s1_tr = Y_tr[:, :9]   # (N_train, 9)
Y_s1_va = Y_va[:, :9]   # (N_val,   9)
print(f"Y_s1 train: {Y_s1_tr.shape}  |  val: {Y_s1_va.shape}")

In [ ]:
# Distribuzione di ciascuna variabile (train vs val)
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for i, (name, ax) in enumerate(zip(S1_VARS, axes)):
    kw = dict(bins=60, edgecolor='white', linewidth=0.3, alpha=0.7, density=True)
    ax.hist(Y_s1_tr[:, i], **kw, color='steelblue', label='train')
    ax.hist(Y_s1_va[:, i], **kw, color='coral',     label='val')
    mu, sig = Y_s1_tr[:, i].mean(), Y_s1_tr[:, i].std()
    ax.set_title(name, fontsize=9)
    ax.text(0.97, 0.95, f'μ={mu:.3g}\nσ={sig:.3g}',
            transform=ax.transAxes, ha='right', va='top', fontsize=7.5, color='navy')
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

plt.suptitle('Stage 1 — distribuzione delle 9 variabili  (train vs val)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Matrice di correlazione tra le 9 variabili di stage 1
corr = np.corrcoef(Y_s1_tr.T)   # (9, 9)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(9)); ax.set_xticklabels(S1_VARS, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(9)); ax.set_yticklabels(S1_VARS, fontsize=8)
for i in range(9):
    for j in range(9):
        ax.text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center',
                fontsize=7, color='black' if abs(corr[i,j]) < 0.7 else 'white')
plt.colorbar(im, ax=ax, label='correlazione di Pearson')
ax.set_title('Stage 1 — matrice di correlazione', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Correlazione tra parametri di input (X[9:25]) e variabili di stage 1
param_names = x_cols[9:]   # 16 parametri di controllo
X_params = X_tr[:, 9:]     # (N, 16)

corr_xy = np.array([[np.corrcoef(X_params[:, p], Y_s1_tr[:, v])[0, 1]
                      for v in range(9)]
                     for p in range(16)])   # (16, 9)

fig, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(corr_xy, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(9));  ax.set_xticklabels(S1_VARS,    rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(16)); ax.set_yticklabels(param_names, fontsize=8)
for p in range(16):
    for v in range(9):
        c = corr_xy[p, v]
        ax.text(v, p, f'{c:.2f}', ha='center', va='center',
                fontsize=6.5, color='black' if abs(c) < 0.6 else 'white')
plt.colorbar(im, ax=ax, label='correlazione di Pearson')
ax.set_title('Correlazione parametri di input → variabili stage 1', fontsize=11)
plt.tight_layout()
plt.show()